## Google Maps & Places API Integration Test

Testing the Google Maps Distance Matrix API and Google Places API — these will power the `get_commute_time` and `find_nearby_places` tools in our listing agent pipeline.

Both tools are currently stubs (see `graph/tools/commute.py` and `graph/tools/places.py`). This notebook validates the APIs, explores response data, and drafts working implementations.

**One API key for everything:** The `googlemaps` Python client uses a single key for Distance Matrix, Geocoding, and Places Nearby Search. We only need `GOOGLE_MAPS_API_KEY` in `.env` — the two separate keys in `.env.example` should be consolidated.

In [ ]:
!pip install googlemaps python-dotenv

In [ ]:
import googlemaps
import json
import os
import math
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_MAPS_API_KEY")
if api_key:
    print(f"GOOGLE_MAPS_API_KEY loaded: {api_key[:4]}...{api_key[-4:]}")
else:
    print("WARNING: GOOGLE_MAPS_API_KEY not found in .env")
    print("Add GOOGLE_MAPS_API_KEY=your_key to your .env file")
    print("Enable Distance Matrix, Places, and Geocoding APIs in your GCP project")

gmaps = googlemaps.Client(key=api_key)
print("googlemaps client initialized")

### Test 1: Distance Matrix API — raw response

Mimics `graph/tools/commute.py` — listing agents call this to get real commute times to destinations from `PreferenceState.commute_destinations` (see `graph/state.py`).

The Distance Matrix API returns actual travel time and distance for a given origin-destination pair and travel mode. This replaces guessing commute times from search snippets.

In [ ]:
origin = "4521 Telegraph Ave, Oakland, CA"
destination = "UC Berkeley, Soda Hall"

try:
    result = gmaps.distance_matrix(origin, destination, mode="driving")
    print("Raw API response:")
    print(json.dumps(result, indent=2))

    print("\n--- Extracted fields ---")
    element = result["rows"][0]["elements"][0]
    if element["status"] == "OK":
        print(f"Duration: {element['duration']['text']} ({element['duration']['value']} seconds)")
        print(f"Distance: {element['distance']['text']}")
    else:
        print(f"Status: {element['status']}")
except Exception as e:
    print(f"API call failed: {e}")
    print("Make sure GOOGLE_MAPS_API_KEY is set and Distance Matrix API is enabled")

### Test 2: All travel modes

Berkeley/Oakland renters care about biking and transit, not just driving. The current stub in `graph/tools/commute.py` lists `transit`, `driving`, `walking` — but `bicycling` is critical for this area and the API supports it.

In [ ]:
origin = "4521 Telegraph Ave, Oakland, CA"
destination = "UC Berkeley, Soda Hall"
modes = ["driving", "transit", "bicycling", "walking"]

print(f"From: {origin}")
print(f"To:   {destination}\n")

for mode in modes:
    try:
        result = gmaps.distance_matrix(origin, destination, mode=mode)
        element = result["rows"][0]["elements"][0]
        if element["status"] == "OK":
            duration = element["duration"]["text"]
            distance = element["distance"]["text"]
            print(f"  {mode:10s}  {duration:>12s}  ({distance})")
        else:
            print(f"  {mode:10s}  {element['status']}")
    except Exception as e:
        print(f"  {mode:10s}  ERROR: {e}")

### Test 3: Multiple destinations in one call

A user might list both a work commute and a gym destination. The API handles multiple destinations in a single request — cheaper than separate calls (billed per origin × destination element).

In [ ]:
origin = "4521 Telegraph Ave, Oakland, CA"
destinations = [
    "UC Berkeley, Soda Hall",
    "Whole Foods, 230 Bay Pl, Oakland, CA",
]

try:
    result = gmaps.distance_matrix(origin, destinations, mode="transit")

    print(f"Origin: {origin}\n")
    for i, dest in enumerate(destinations):
        element = result["rows"][0]["elements"][i]
        if element["status"] == "OK":
            print(f"  \u2192 {dest}")
            print(f"    {element['duration']['text']} ({element['distance']['text']})\n")
        else:
            print(f"  \u2192 {dest}: {element['status']}\n")
except Exception as e:
    print(f"API call failed: {e}")

### Test 4: Proposed return format for `get_commute_time`

The current stub returns `str`, but every other tool in `graph/tools/` returns `dict` or `list[dict]`. Building the structured format the listing agent should receive.

In [ ]:
def build_commute_result(origin, destination, mode, api_response):
    """Extract the fields our listing agent needs from a Distance Matrix response."""
    element = api_response["rows"][0]["elements"][0]
    if element["status"] != "OK":
        return {
            "error": f"No route found: {element['status']}",
            "origin": origin,
            "destination": destination,
            "mode": mode,
        }
    return {
        "origin": origin,
        "destination": destination,
        "mode": mode,
        "duration_text": element["duration"]["text"],
        "duration_seconds": element["duration"]["value"],
        "distance_text": element["distance"]["text"],
    }

try:
    result = gmaps.distance_matrix(
        "4521 Telegraph Ave, Oakland, CA",
        "UC Berkeley, Soda Hall",
        mode="bicycling",
    )
    commute = build_commute_result(
        "4521 Telegraph Ave, Oakland, CA", "UC Berkeley, Soda Hall", "bicycling", result
    )
    print("Proposed return format:")
    print(json.dumps(commute, indent=2))
except Exception as e:
    print(f"API call failed: {e}")

### Test 5: Distance Matrix edge cases

In [ ]:
# Invalid address
print("--- Invalid address ---")
try:
    result = gmaps.distance_matrix("zzz not a real place 12345", "UC Berkeley", mode="driving")
    element = result["rows"][0]["elements"][0]
    print(f"Status: {element['status']}")
except Exception as e:
    print(f"Exception: {e}")

# No route possible (ocean crossing by transit)
print("\n--- No transit route (Honolulu \u2192 Berkeley) ---")
try:
    result = gmaps.distance_matrix("Honolulu, HI", "UC Berkeley, CA", mode="transit")
    element = result["rows"][0]["elements"][0]
    print(f"Status: {element['status']}")
except Exception as e:
    print(f"Exception: {e}")

**Key Takeaways — Distance Matrix API**

- All four modes (`driving`, `transit`, `bicycling`, `walking`) work for Berkeley/Oakland. The stub should support `bicycling` in addition to the current three.
- Batch requests work — multiple destinations in a single call reduce API overhead.
- Invalid addresses return `status: "NOT_FOUND"`, unreachable routes return `status: "ZERO_RESULTS"`. Both are handled gracefully without exceptions.
- The tool should return a structured `dict`, not a `str`, matching the pattern in `search_web` and `scrape_listing`.

**Expected output format:**
```json
{
  "origin": "4521 Telegraph Ave, Oakland, CA",
  "destination": "UC Berkeley, Soda Hall",
  "mode": "bicycling",
  "duration_text": "12 mins",
  "duration_seconds": 720,
  "distance_text": "2.1 mi"
}
```

### Test 6: Geocoding — address to coordinates

The Places Nearby Search requires lat/lng, but our listing agents only have street addresses. Geocoding converts addresses to coordinates — this will be an internal step inside `find_nearby_places`, not a separate tool.

The `googlemaps` client handles geocoding through the same API key.

In [ ]:
try:
    # Primary test address
    address = "4521 Telegraph Ave, Oakland, CA"
    geocode_result = gmaps.geocode(address)

    location = geocode_result[0]["geometry"]["location"]
    lat, lng = location["lat"], location["lng"]
    print(f"Address:     {address}")
    print(f"Coordinates: ({lat}, {lng})")
    print(f"Formatted:   {geocode_result[0]['formatted_address']}")

    # Robustness: partial address
    print("\n--- Partial address ---")
    partial = gmaps.geocode("Telegraph Ave, Oakland")
    if partial:
        loc = partial[0]["geometry"]["location"]
        print(f"'Telegraph Ave, Oakland' \u2192 ({loc['lat']}, {loc['lng']})")
        print(f"  Resolved to: {partial[0]['formatted_address']}")

    # Robustness: neighborhood name
    print("\n--- Neighborhood name ---")
    neighborhood = gmaps.geocode("Temescal, Oakland, CA")
    if neighborhood:
        loc = neighborhood[0]["geometry"]["location"]
        print(f"'Temescal, Oakland, CA' \u2192 ({loc['lat']}, {loc['lng']})")
        print(f"  Resolved to: {neighborhood[0]['formatted_address']}")
    else:
        print("'Temescal, Oakland, CA' \u2014 no results")

except Exception as e:
    print(f"Geocoding failed: {e}")

### Test 7: Places Nearby Search — finding amenities

Mimics `graph/tools/places.py` — listing agents call this to find nearby gyms, bars, grocery stores, etc. based on what the user mentioned in their `soft_constraints`.

Uses the coordinates from geocoding above. Searching for gyms within 1000m (reasonable walking distance).

In [ ]:
try:
    places_result = gmaps.places_nearby(
        location=(lat, lng),
        radius=1000,
        type="gym",
    )

    results = places_result.get("results", [])
    print(f"Found {len(results)} gyms within 1000m of ({lat}, {lng})\n")

    if results:
        print("Raw first result:")
        print(json.dumps(results[0], indent=2))
except Exception as e:
    print(f"Places search failed: {e}")
    print("Make sure Places API is enabled in your GCP project")

In [ ]:
def haversine_distance(lat1, lng1, lat2, lng2):
    """Distance in meters between two lat/lng points."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

print("Gyms near 4521 Telegraph Ave, Oakland:\n")
for place in places_result.get("results", [])[:5]:
    ploc = place["geometry"]["location"]
    dist = haversine_distance(lat, lng, ploc["lat"], ploc["lng"])
    print(f"  {place['name']}")
    print(f"    Address:  {place.get('vicinity', 'N/A')}")
    print(f"    Rating:   {place.get('rating', 'N/A')} ({place.get('user_ratings_total', 0)} reviews)")
    print(f"    Distance: {dist:.0f}m")
    print()

### Test 8: Multiple place types

Testing the types most relevant to renters. These map to what users mention in `soft_constraints` — "gym nearby", "bars within walking distance", "close to grocery stores".

In [ ]:
place_types = ["gym", "bar", "supermarket", "park", "restaurant"]

print(f"Searching near 4521 Telegraph Ave, Oakland (1000m radius)\n")

for ptype in place_types:
    try:
        result = gmaps.places_nearby(location=(lat, lng), radius=1000, type=ptype)
        places = result.get("results", [])
        if places:
            top = places[0]
            dist = haversine_distance(
                lat, lng,
                top["geometry"]["location"]["lat"],
                top["geometry"]["location"]["lng"],
            )
            print(f"  {ptype:15s}  {len(places):2d} results  (nearest: {top['name']}, {dist:.0f}m)")
        else:
            print(f"  {ptype:15s}   0 results")
    except Exception as e:
        print(f"  {ptype:15s}  ERROR: {e}")

### Test 9: Proposed return format + no-results edge case

Building the structured dict that `find_nearby_places` should return, then testing what happens when searching for a place type unlikely to exist in a residential area.

In [ ]:
def build_places_result(address, place_type, radius, origin_lat, origin_lng, api_results):
    """Build the structured response for the listing agent."""
    results = []
    for place in api_results[:5]:
        ploc = place["geometry"]["location"]
        dist = haversine_distance(origin_lat, origin_lng, ploc["lat"], ploc["lng"])
        results.append({
            "name": place["name"],
            "address": place.get("vicinity", ""),
            "rating": place.get("rating"),
            "total_ratings": place.get("user_ratings_total", 0),
            "distance_meters": round(dist),
        })
    return {
        "query_type": place_type,
        "address": address,
        "radius_meters": radius,
        "results_count": len(api_results),
        "results": results,
    }

# Demo with gym results from Test 7
places_formatted = build_places_result(
    "4521 Telegraph Ave, Oakland, CA", "gym", 1000, lat, lng,
    places_result.get("results", []),
)
print("Proposed return format:")
print(json.dumps(places_formatted, indent=2))

# Edge case: no results expected
print("\n--- Edge case: amusement_park (residential area) ---")
try:
    edge_result = gmaps.places_nearby(location=(lat, lng), radius=1000, type="amusement_park")
    edge_places = edge_result.get("results", [])
    print(f"Results: {len(edge_places)}")
    if not edge_places:
        print("No amusement parks within 1000m \u2014 expected for a residential area")
except Exception as e:
    print(f"Error: {e}")

**Key Takeaways — Geocoding + Places Nearby Search**

- Geocoding handles full addresses reliably. Partial addresses and neighborhood names resolve but may point to unexpected locations — always use the full listing address.
- Nearby Search returns structured data: name, address, rating, review count, coordinates. Combined with haversine distance, it gives the listing agent exactly what it needs.
- Common types (`gym`, `bar`, `supermarket`, `park`, `restaurant`) return good results in urban areas.
- Rare types (`amusement_park`) return empty results gracefully — no exception, just an empty list.
- The tool should geocode internally (address in, structured results out) so listing agents don't manage coordinates.

**Expected output format:**
```json
{
  "query_type": "gym",
  "address": "4521 Telegraph Ave, Oakland, CA",
  "radius_meters": 1000,
  "results_count": 8,
  "results": [
    {
      "name": "Planet Fitness",
      "address": "123 Broadway, Oakland",
      "rating": 4.2,
      "total_ratings": 340,
      "distance_meters": 450
    }
  ]
}
```

### Cost and Rate Limit Analysis

Pricing matters because our listing agents decide which tools to call — if a user doesn't mention gyms, the agent skips the places lookup and saves money.

| API | Cost | Free tier |
|-----|------|-----------|
| Distance Matrix | $5.00 per 1,000 elements (origin \u00d7 destination pair) | $200/month credit |
| Places Nearby Search | $32.00 per 1,000 requests | $200/month credit |
| Geocoding | $5.00 per 1,000 requests | $200/month credit |

Rate limits: 50 requests/second for Distance Matrix, 60K queries/day default for Places.

In [ ]:
geocode_calls = 1
nearby_searches = 2       # average: user mentions ~2 place types
commute_lookups = 1.5     # average: 1-2 commute destinations

geocode_cost = geocode_calls * (5.00 / 1000)
places_cost = nearby_searches * (32.00 / 1000)
commute_cost = commute_lookups * (5.00 / 1000)
per_listing = geocode_cost + places_cost + commute_cost

total_listings = 25  # candidates processed before filtering to ~10-15
session_cost = per_listing * total_listings

print("Cost per listing agent run:")
print(f"  Geocoding:       ${geocode_cost:.4f}  ({geocode_calls} call)")
print(f"  Nearby Search:   ${places_cost:.4f}  ({nearby_searches} calls avg)")
print(f"  Distance Matrix: ${commute_cost:.4f}  ({commute_lookups} calls avg)")
print(f"  Total:           ${per_listing:.4f}")
print()
print(f"Cost per session ({total_listings} listings processed):")
print(f"  ${session_cost:.2f}")
print()
print(f"$200/month free credit covers ~{200 / session_cost:.0f} sessions")
print()
print("Agents that disqualify listings early (over budget, no pets) skip")
print("location API calls entirely, so real cost is lower than this estimate.")

**Key Takeaways — Cost**

- Per-listing cost is very low (~$0.07 with typical tool usage).
- A full session processing 25 listings costs ~$1.84.
- The $200/month free tier covers 100+ sessions — plenty for development and moderate usage.
- The ReAct architecture helps: agents that disqualify listings early skip location API calls, and agents only call `find_nearby_places` for place types the user actually mentioned.

### Prototype Tool Implementations

Drop-in replacements for the stubs in `graph/tools/commute.py` and `graph/tools/places.py`. These follow the same pattern used by `search_web`, `scrape_listing`, and `analyze_listing_photos`:

```python
from langchain_core.tools import tool

@tool
async def tool_name(param: str) -> dict:
    ...
```

**Breaking change from stubs:** Return type is `dict` instead of `str`. Since the stubs aren't used in production yet, this is safe.

In [ ]:
# Prototype: get_commute_time
# In production: add @tool decorator, make async, move to graph/tools/commute.py

def get_commute_time_v2(origin: str, destination: str, mode: str = "transit") -> dict:
    """
    Get commute time between two addresses.
    Modes: 'driving', 'transit', 'bicycling', 'walking'.
    """
    client = googlemaps.Client(key=os.getenv("GOOGLE_MAPS_API_KEY"))
    try:
        result = client.distance_matrix(origin, destination, mode=mode)
        element = result["rows"][0]["elements"][0]
        if element["status"] != "OK":
            return {
                "error": f"No route found ({element['status']})",
                "origin": origin,
                "destination": destination,
                "mode": mode,
            }
        return {
            "origin": origin,
            "destination": destination,
            "mode": mode,
            "duration_text": element["duration"]["text"],
            "duration_seconds": element["duration"]["value"],
            "distance_text": element["distance"]["text"],
        }
    except Exception as e:
        return {"error": str(e), "origin": origin, "destination": destination, "mode": mode}

# Test
print("get_commute_time_v2:")
print(json.dumps(get_commute_time_v2("4521 Telegraph Ave, Oakland, CA", "UC Berkeley, Soda Hall", "bicycling"), indent=2))

In [ ]:
# Prototype: find_nearby_places
# In production: add @tool decorator, make async, move to graph/tools/places.py
# Include haversine_distance in the tool file or a shared utils module

def find_nearby_places_v2(address: str, place_type: str, radius: int = 1000) -> dict:
    """
    Find places of a given type near an address.
    Geocodes the address internally, then searches within radius (meters).
    place_type examples: 'gym', 'bar', 'supermarket', 'park', 'restaurant'.
    """
    client = googlemaps.Client(key=os.getenv("GOOGLE_MAPS_API_KEY"))
    try:
        geo = client.geocode(address)
        if not geo:
            return {
                "error": f"Could not geocode address: {address}",
                "query_type": place_type,
                "address": address,
            }

        loc = geo[0]["geometry"]["location"]
        origin_lat, origin_lng = loc["lat"], loc["lng"]

        places = client.places_nearby(
            location=(origin_lat, origin_lng), radius=radius, type=place_type
        )

        results = []
        for place in places.get("results", [])[:5]:
            ploc = place["geometry"]["location"]
            dist = haversine_distance(origin_lat, origin_lng, ploc["lat"], ploc["lng"])
            results.append({
                "name": place["name"],
                "address": place.get("vicinity", ""),
                "rating": place.get("rating"),
                "total_ratings": place.get("user_ratings_total", 0),
                "distance_meters": round(dist),
            })

        return {
            "query_type": place_type,
            "address": address,
            "radius_meters": radius,
            "results_count": len(places.get("results", [])),
            "results": results,
        }
    except Exception as e:
        return {"error": str(e), "query_type": place_type, "address": address}

# Test
print("find_nearby_places_v2:")
print(json.dumps(find_nearby_places_v2("4521 Telegraph Ave, Oakland, CA", "gym"), indent=2))

In [ ]:
# Smoke test: both tools together, simulating a listing agent run
address = "2090 Kittredge St, Berkeley, CA"

print(f"=== Listing: {address} ===\n")

# Commute check (user listed UC Berkeley as destination)
commute = get_commute_time_v2(address, "UC Berkeley, Soda Hall", mode="transit")
print(f"Commute to Soda Hall: {commute.get('duration_text', commute.get('error'))}")

# Nearby places (user mentioned gym in soft_constraints)
gyms = find_nearby_places_v2(address, "gym")
if gyms.get("results"):
    nearest = gyms["results"][0]
    print(f"Nearest gym: {nearest['name']} ({nearest['distance_meters']}m)")
else:
    print(f"Gyms: {gyms.get('error', 'no results')}")

# Nearby places (user mentioned bars in soft_constraints)
bars = find_nearby_places_v2(address, "bar")
if bars.get("results"):
    nearest = bars["results"][0]
    print(f"Nearest bar: {nearest['name']} ({nearest['distance_meters']}m)")
else:
    print(f"Bars: {bars.get('error', 'no results')}")

### Summary

**What works:**
- Distance Matrix API returns accurate commute times for all four modes (`driving`, `transit`, `bicycling`, `walking`). Bicycling mode is well-supported in the Bay Area and should be added to the tool's supported modes.
- Places Nearby Search returns structured amenity data (name, rating, reviews, coordinates). Combined with haversine distance, it gives the listing agent exactly what it needs.
- Geocoding is reliable for full street addresses. The tool should always use the full listing address, not partial or neighborhood names.
- A single `googlemaps` client with one API key handles all three APIs.

**Action items:**
1. Add `googlemaps>=4.10.0` to `requirements.txt`
2. Consolidate `.env.example` — replace the two commented-out Google keys with a single `GOOGLE_MAPS_API_KEY` (covers Distance Matrix + Places + Geocoding)
3. Replace stubs in `graph/tools/commute.py` and `graph/tools/places.py` with the prototype implementations above (change return type from `str` to `dict`)
4. Update `get_commute_time` docstring to include `bicycling` as a supported mode
5. Add `haversine_distance` as a utility function (either in the tool file or a shared `graph/tools/utils.py`)